In [1]:
import sys
sys.path.insert(0, '/home/scanbot/ua_tensors')

import numpy as np
import matplotlib.pyplot as plt
from embed import Embed
import requests

T = Embed(temp=0.0)

from attention import Attention
from algebra import Abs, Sum

In [2]:
url  = "https://raw.githubusercontent.com/mledoze/countries/master/countries.json"
data = requests.get(url).json()

# --- build entity lists ---
countries, regions, subregions, languages = [], [], [], []
triples = []

cca3_to_name = {e['cca3']: e.get('name', {}).get('common', '') for e in data}

for entry in data:
    country   = entry.get('name', {}).get('common', '')
    region    = entry.get('region', '')
    subregion = entry.get('subregion', '')
    langs     = list(entry.get('languages', {}).values())
    borders   = entry.get('borders', [])  # list of cca3 codes
    currencies = list(entry.get('currencies', {}).keys())
    timezones  = entry.get('timezones', [])

    if not (country and region):
        continue
    countries.append(country)
    regions.append(region)
    if subregion:
        subregions.append(subregion)
    languages.extend(langs)
    triples.append((country, 'region',    region))
    triples.append((country, 'subregion', subregion) if subregion else None)
    for lang in langs:
        triples.append((country, 'speaks', lang))
    for b in borders:
        triples.append((country, 'borders', cca3_to_name.get(b, '')))
    for c in currencies:
        triples.append((country, 'currency', c))
    for tz in timezones:
        triples.append((country, 'timezone', tz))

triples = [t for t in triples if t]

countries   = sorted(set(countries))
regions     = sorted(set(regions))
subregions  = sorted(set(subregions))
languages   = sorted(set(languages))
borders_list  = sorted(set(countries))   # country→country
currencies    = sorted(set(t[2] for t in triples if t[1] == 'currency'))
timezones     = sorted(set(t[2] for t in triples if t[1] == 'timezone'))

c_idx = {c: i for i, c in enumerate(countries)}
r_idx = {r: i for i, r in enumerate(regions)}
s_idx = {s: i for i, s in enumerate(subregions)}
l_idx = {l: i for i, l in enumerate(languages)}
cu_idx = {c: i for i, c in enumerate(currencies)}
tz_idx = {t: i for i, t in enumerate(timezones)}

NC, NR, NS, NL = len(countries), len(regions), len(subregions), len(languages)
print(f"Countries: {NC}, Regions: {NR}, Subregions: {NS}, Languages: {NL}")

# --- build relation matrices ---
R_loc    = np.zeros((NC, NR))   # country → region   (binary)
R_sub    = np.zeros((NC, NS))   # country → subregion
R_speaks = np.zeros((NC, NL))   # country → language
R_borders  = np.zeros((NC, NC))    # country × country
R_currency = np.zeros((NC, len(currencies)))
R_timezone = np.zeros((NC, len(timezones)))

for t in triples:
    country, rel, target = t
    if country not in c_idx:
        continue
    ci = c_idx[country]
    if rel == 'region'    and target in r_idx: R_loc[ci,    r_idx[target]] = 1.0
    if rel == 'subregion' and target in s_idx: R_sub[ci,    s_idx[target]] = 1.0
    if rel == 'speaks'    and target in l_idx: R_speaks[ci, l_idx[target]] = 1.0
    if rel == 'borders'  and target in c_idx:  R_borders[ci,  c_idx[target]]  = 1.0
    if rel == 'currency' and target in cu_idx: R_currency[ci, cu_idx[target]] = 1.0
    if rel == 'timezone' and target in tz_idx: R_timezone[ci, tz_idx[target]] = 1.0

print(f"R_loc    shape: {R_loc.shape}   density: {R_loc.mean():.3f}")
print(f"R_sub    shape: {R_sub.shape}  density: {R_sub.mean():.3f}")
print(f"R_speaks shape: {R_speaks.shape}  density: {R_speaks.mean():.3f}")

Countries: 250, Regions: 6, Subregions: 24, Languages: 155
R_loc    shape: (250, 6)   density: 0.167
R_sub    shape: (250, 24)  density: 0.041
R_speaks shape: (250, 155)  density: 0.011


In [3]:
# Updated Cell 4 logic to ensure 2D outputs
emb_loc,      EmbR_loc,      rep_loc      = T.ConceptEmbed(R_loc,      temp=0.1)
emb_sub,      EmbR_sub,      rep_sub      = T.ConceptEmbed(R_sub,      temp=0.1)
emb_speaks,   EmbR_speaks,   rep_speaks   = T.ConceptEmbed(R_speaks,   temp=0.1)
emb_borders,  EmbR_borders,  rep_borders  = T.ConceptEmbed(R_borders,  temp=0.1)
emb_currency, EmbR_currency, rep_currency = T.ConceptEmbed(R_currency, temp=0.1)
emb_timezone, EmbR_timezone, rep_timezone = T.ConceptEmbed(R_timezone, temp=0.1)

# FORCE RE-SHAPE: This ensures that even if a variable came back 1D, 
# it is forced back into a 2D column/matrix so the downstream cells don't crash.
emb_loc      = np.atleast_2d(emb_loc)
emb_sub      = np.atleast_2d(emb_sub)
emb_speaks   = np.atleast_2d(emb_speaks)
emb_borders  = np.atleast_2d(emb_borders)
emb_currency = np.atleast_2d(emb_currency)
emb_timezone = np.atleast_2d(emb_timezone)

# Now your original horizontal stack will also be safe
emb = np.hstack([emb_loc, emb_sub, emb_speaks, emb_borders, emb_currency, emb_timezone])

d0 = emb_loc.shape[1]
d1 = d0 + emb_sub.shape[1]
d2 = d1 + emb_speaks.shape[1]
d3 = d2 + emb_borders.shape[1]
d4 = d3 + emb_currency.shape[1]
d5 = d4 + emb_timezone.shape[1]

head_cols = [
    np.arange(0,  d0),
    np.arange(d0, d1),
    np.arange(d1, d2),
    np.arange(d2, d3),
    np.arange(d3, d4),
    np.arange(d4, d5),
]

concept_labels = (
    [('loc',      regions[c])    for c in rep_loc]      +
    [('sub',      subregions[c]) for c in rep_sub]      +
    [('speaks',   languages[c])  for c in rep_speaks]   +
    [('borders',  countries[c])  for c in rep_borders]  +
    [('currency', currencies[c]) for c in rep_currency] +
    [('timezone', timezones[c])  for c in rep_timezone]
)

/usr/local/lib/python3.11/dist-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.11/dist-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [4]:
mem_loc      = Attention(emb_loc,      temp=0.9)
mem_sub      = Attention(emb_sub,      temp=0.9)
mem_speaks   = Attention(emb_speaks,   temp=0.9)

In [5]:
def decode(mha, labels):
    extent = sorted([(countries[i], mha.fp.state[i]) for i in np.where(mha.fp.state > mha.eps)[0]], key=lambda x: -x[1])
    print("Extent:", [(c, f"{s:.3f}") for c, s in extent])
    for name, (intent, _) in mha.intents.items():
        label_arr, rep_cols = labels[name]
        concepts = sorted([(label_arr[rep_cols[i]], intent[i]) for i in range(len(rep_cols)) if intent[i] > mha.eps], key=lambda x: -x[1])
        print(f"Intent [{name}]:", [(c, f"{s:.3f}") for c, s in concepts])

In [6]:
def decode(mha, labels):
    extent = sorted([(countries[i], mha.fp.state[i]) for i in np.where(mha.fp.state > mha.eps)[0]], key=lambda x: -x[1])
    print("Extent:", [(c, f"{s:.3f}") for c, s in extent])
    for name, (intent, _) in mha.intents.items():
        label_arr, rep_cols = labels[name]
        concepts = sorted([(label_arr[rep_cols[i]], intent[i]) for i in range(len(rep_cols)) if intent[i] > mha.eps], key=lambda x: -x[1])
        print(f"Intent [{name}]:", [(c, f"{s:.3f}") for c, s in concepts])

In [7]:
from functools import reduce
from attention import MultiHeadAttention
import numpy as np

mha = MultiHeadAttention(
    heads = [mem_loc, mem_sub, mem_speaks],
    names = ['loc', 'sub', 'speaks'],
)

idx = [c_idx["Pakistan"], c_idx["Albania"]]
hits, intents = mha.retrieve(idx)
print()
decode(mha, {
    'loc':    (regions, rep_loc),
    'sub':    (subregions, rep_sub),
    'speaks': (languages, rep_speaks),
})


Extent: [('Albania', '1.000'), ('Pakistan', '1.000')]


In [12]:
from fixpoint import FixpointIterator

class CategoryExplorer:
    def __init__(self, mha, eps=1e-3):
        self.mha = mha
        self.eps = eps
        self.categories = {}

    def _intent_key(self):
        parts = []
        for name, (intent, _) in self.mha.intents.items():
            parts.append(tuple((intent / self.eps).astype(int)))
        return tuple(parts)
    
    # select a nonzero index from the extent vector of a category
    def _representative(self, extent):
        nonzero = np.where(extent > self.eps)[0]
        if len(nonzero) == 0:
            return None
        return nonzero[np.argmin(extent[nonzero])]
    
    # check if a relationship between two extents has already been discovered
    def _co_occurs(self, i, j):
        return any(
            cat['extent'][i] > self.eps and cat['extent'][j] > self.eps
            for cat in self.categories.values()
        )

    # Perform an initial exploration 
    def explore(self, n_entities):
        for i in range(n_entities):
            hits, intents = self.mha.retrieve([i])
            if len(hits) == 0:
                continue
            key = self._intent_key()
            if key not in self.categories:
                self.categories[key] = {
                    'intents': dict(self.mha.intents), 
                    'extent': self.mha.fp.state.copy()
                }
        return self.categories
    
    def _lattice_step(self, state, temp):
        keys = list(self.categories.keys())
        for a, key_a in enumerate(keys):
            for key_b in keys[a+1:]:
                i = self._representative(self.categories[key_a]['extent'])
                j = self._representative(self.categories[key_b]['extent'])
                if i is None or j is None:
                    continue
                if self._co_occurs(i, j):
                    continue
                hits, _ = self.mha.retrieve([i, j])
                if len(hits) == 0:
                    continue
                new_key = self._intent_key()
                if new_key not in self.categories:
                    self.categories[new_key] = {'intents': dict(self.mha.intents), 'extent': self.mha.fp.state.copy()}
        return np.maximum.reduce([cat['extent'] for cat in self.categories.values()]), None
    
    def explore_lattice(self, n_entities):
        self.explore(n_entities)
        union = np.maximum.reduce([cat['extent'] for cat in self.categories.values()])
        fp = FixpointIterator(
            f         = self._lattice_step,
            energy_fn = lambda new, old, aux: float(Sum(Abs(new - old))),
            state0    = union,
            eps       = self.eps,
            max_iters = 50,
        )
        fp.run()
        # Prune categories with empty intents - where no relationship was found
        self.categories = {
            k: v for k, v in self.categories.items() 
            if v['intents'] and any(np.any(intent > self.eps) for _, (intent, _) in v['intents'].items())
        }
        print(f"Valid categories: {len(self.categories)}")
        return self.categories

    def isomorphic_groups(self):
        groups = {}
        for key, cat in self.categories.items():
            iso_key = tuple(
                tuple(sorted(intent[intent > self.eps], reverse=True))
                for _, (intent, _) in cat['intents'].items()
            )
            if iso_key not in groups:
                groups[iso_key] = []
            groups[iso_key].append(key)
        return groups

In [13]:
explorer = CategoryExplorer(mha)
categories = explorer.explore_lattice(len(countries))
print(f"Unique categories found: {len(categories)}")

Valid categories: 176
Unique categories found: 176


In [15]:
groups = explorer.isomorphic_groups()
print(f"Isomorphic groups: {len(groups)}")
for iso_key, members in groups.items():
    print(f"Group size {len(members)}: {iso_key}")

Isomorphic groups: 146
Group size 1: ((np.float64(0.99999781467763),), (np.float64(0.39999999990757684), np.float64(0.2000000000888433), np.float64(0.1999999999468716), np.float64(0.1999999999468716)), (np.float64(0.9999701757550893),))
Group size 1: ((np.float64(0.9999979538167193),), (np.float64(0.6666666666666701), np.float64(0.33333333333333376)), (np.float64(0.9999701757550893),))
Group size 1: ((np.float64(0.9999981835151672),), (np.float64(0.6578947368421169), np.float64(0.10526315789473734), np.float64(0.05263157894736918), np.float64(0.02631578947368446), np.float64(0.02631578947368446), np.float64(0.02631578947368446), np.float64(0.02631578947368446), np.float64(0.02631578947368446), np.float64(0.02631578947368446), np.float64(0.02631578947368446)), (np.float64(0.999958196524573),))
Group size 1: ((np.float64(0.9999952845284305),), (np.float64(0.48924731182795106), np.float64(0.04838709677419298), np.float64(0.021505376344085957), np.float64(0.0161290322580644), np.float64(0.

In [17]:
non_singletons = {k: v for k, v in groups.items() if len(v) > 1}
print(f"Non-singleton isomorphic groups: {len(non_singletons)}")

Non-singleton isomorphic groups: 17


In [18]:
for iso_key, members in groups.items():
    if len(members) > 1:
        for key in members:
            mha.intents = explorer.categories[key]['intents']
            decode(mha, {
                'loc':    (regions, rep_loc),
                'sub':    (subregions, rep_sub),
                'speaks': (languages, rep_speaks),
            })
        print("---")

Extent: [('Malaysia', '0.489'), ('Philippines', '0.489'), ('Singapore', '0.489'), ('Brunei', '0.011')]
Intent [loc]: [('Asia', '1.000')]
Intent [speaks]: [('Armenian', '0.994')]
Intent [sub]: [('Western Asia', '1.000')]
Extent: [('Malaysia', '0.489'), ('Philippines', '0.489'), ('Singapore', '0.489'), ('Brunei', '0.011')]
Intent [loc]: [('Asia', '1.000')]
Intent [speaks]: [('Georgian', '0.994')]
Intent [sub]: [('Western Asia', '1.000')]
---
Extent: [('Malaysia', '0.489'), ('Philippines', '0.489'), ('Singapore', '0.489'), ('Brunei', '0.011')]
Intent [loc]: [('Oceania', '1.000')]
Intent [speaks]: [('English', '0.489'), ('French', '0.048'), ('Tswana', '0.022'), ('Dutch', '0.016'), ('Samoan', '0.016'), ('Spanish', '0.016'), ('Swahili', '0.016'), ('Afrikaans', '0.011'), ('Arabic', '0.011'), ('Chamorro', '0.011'), ('Chewa', '0.011'), ('Chinese', '0.011'), ('Malay', '0.011'), ('Papiamento', '0.011'), ('Swazi', '0.011'), ('Tamil', '0.011'), ('Tsonga', '0.011'), ('Belizean Creole', '0.005'), ('B